# CS604 Colab launcher — Van: top-k and error causes for the best full fine-tune at 448, on ALL 10,000 classes

Van's Task 1 final recipe (`full_ft_20261017_122237`: CodaBench 0.825 plain, 0.832 crop 0.9 + zoom) has no labelled holdout: it trains on all 45 photos per class and the val labels are hidden. This trains the **same recipe** (`configs/van_full_ft_448_best.yaml`, incl. fused AdamW + torch.compile) for the full **144 training minutes on all 10,000 classes with 5 photos per class held out** (`configs/allclass_holdout_full_ft_448.yaml`, the same `c10000_h5_s0` split as Ana's LoRA run): 400,000 training photos, 50,000 labelled holdout photos the model never sees.

Then, inference only, on that holdout:
1. `scripts/error_analysis.py`: top-1 to top-10, where the truth sits when wrong, errors on relatives, galleries;
2. `scripts/error_causes.py`: the four error flags (sibling, unseen look, two looks, messy class) with a gallery per flag.

Timeline (A100 40 GB): data ~15 min, prototypes at 448 from the fit photos ~30 min, **training 144 min** (+ a few minutes of holdout evaluation per epoch, outside the budget), analysis ~40 min: about **4.5–5 hours**.

**If Colab crashes:** open a new session and **Run all** again. Training resumes from the last finished epoch; finished steps are skipped.

Outputs: `CS604_RUNS/ablation_allclass_holdout_full_ft_448/` (training), `CS604_RUNS/allclass_holdout_ft448_analysis/` (top-k and error causes; Ana's LoRA folder `allclass_holdout_analysis/` is not touched). Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "van-allclass-ft448"
PILOT_DATA_DIR = "/content/pilot_data"
SPEC = "configs/allclass_holdout_full_ft_448.yaml"
FEATURE_DIR = "/content/features_pilot"  # local disk
ARM = "A_all_classes_144min"
OUT_NAME = "allclass_holdout_ft448_analysis"
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Train on all classes, 5 photos per class held out (144 training minutes)

Builds the all-class split (`c10000_h5_s0`), rebuilds prototypes from the fit photos only, then trains. Rerunning resumes an interrupted run.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path


def run(cmd, what):
    proc = subprocess.run([sys.executable, "-u", *cmd])
    if proc.returncode != 0:
        end_runtime(f"{what} failed (exit {proc.returncode}); see the output above")
        raise SystemExit(f"{what} failed")


run(["scripts/ablation.py", "--spec", SPEC, "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
     "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR], "training")
RUN_FOLDER = Path(DRIVE_OUT) / f"ablation_{Path(SPEC).stem}"
record = json.loads((RUN_FOLDER / "arms" / f"{ARM}.json").read_text())
RUN_DIR = record["run_dir"]
PILOT_ROOT = f"{PILOT_DATA_DIR}/c10000_h5_s0"
print("holdout top-1:", record["final_val_acc"], "| near5:", record.get("final_val_acc_near5"), "| run:", RUN_DIR)


## 6. Top-k and error analysis on the 50,000 holdout photos (inference only)

In [ ]:
OUT = Path(DRIVE_OUT) / OUT_NAME
if not (OUT / "errors" / "summary.json").is_file():
    run(["scripts/error_analysis.py", "--pilot_root", PILOT_ROOT, "--data_root", DATA_DIR,
         "--out_dir", str(OUT / "errors"), "--run", f"ft448_all={RUN_DIR}"], "error analysis")
summary = json.loads((OUT / "errors" / "summary.json").read_text())["ft448_all"]
print(json.dumps({k: summary[k] for k in summary if k != "run_dir"}, indent=2))


## 7. Error causes on the same holdout (inference only)


In [ ]:
if not (OUT / "causes" / "causes.json").is_file():
    run(["scripts/error_causes.py", "--run_dir", RUN_DIR, "--pilot_root", PILOT_ROOT,
         "--out_dir", str(OUT / "causes")], "error causes")
causes = json.loads((OUT / "causes" / "causes.json").read_text())
print(json.dumps({k: causes[k] for k in ("errors", "share_of_errors", "same_flag_among_correct_photos",
                                         "share_of_errors_with_no_flag", "thresholds")}, indent=2))


## 8. Figures

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUT / "errors" / "topk.png")))
for png in sorted((OUT / "errors").glob("gallery_ft448_all_*.png"))[:1]:
    display(Image(filename=str(png)))

for png in sorted((OUT / "causes").glob("causes_*.png")):
    display(Image(filename=str(png)))


## 9. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")